In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
df = pd.read_csv('/content/drive/MyDrive/monkeytype/mt_dataset_pref4feat16.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4675 entries, 0 to 4674
Columns: 164 entries, x1_linspace to x4_f15
dtypes: float64(160), int64(4)
memory usage: 5.8 MB


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

def filter_by_participant(df: pd.DataFrame, participant: str):
    """
    participant: 'J' | 'F' | 'Y'
    Фильтруем по x1_participant_* (обычно participant одинаков для x1..x4).
    """
    col = f"x1_participant_{participant}"
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")
    return df[df[col] == 1].copy()

@torch.no_grad()
def eval_test_by_participants_classifier(
    df_test: pd.DataFrame,
    model: nn.Module,
    scaler: StandardScaler,
    X_cols: list[str],
    y_cols: list[str],
    device: str,
    *,
    batch_size: int = 256,
):
    """
    Возвращает DataFrame с метриками loss/acc по каждому участнику + overall.
    """
    rows = []

    def eval_on_df(df_part, name):
        Xp, yp = make_X_y(df_part, X_cols, y_cols)
        Xp = scaler.transform(Xp).astype(np.float32)
        loader = DataLoader(TabDataset(Xp, yp), batch_size=batch_size, shuffle=False)
        m = eval_loop(model, loader, device)
        rows.append({
            "participant": name,
            "n": len(df_part),
            "loss": m["loss"],
            "acc": m["acc"],
        })

    # overall
    eval_on_df(df_test, "ALL")

    # per participant
    for p in ["J", "F", "Y"]:
        df_p = filter_by_participant(df_test, p)
        if len(df_p) == 0:
            rows.append({"participant": p, "n": 0, "loss": np.nan, "acc": np.nan})
        else:
            eval_on_df(df_p, p)

    return pd.DataFrame(rows)

In [15]:
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

feat_names = [
    #"linspace",
    "linspace_session_id",
    "linspace_within_session",
    #"brightness",
    "time_reaction", "omission_percent",
    #"symbol_m", "symbol_c", "symbol_s", "symbol_y", "symbol_f", "symbol_j",
    #"location_0", "location_1", "location_2", "location_3",
    "participant_J", "participant_F", "participant_Y",
    #"correct_symbol_m", "correct_symbol_c", "correct_symbol_s", "correct_symbol_y", "correct_symbol_f", "correct_symbol_j",
]
latent_dim = 16
feat_names_fs = [f"f{i}" for i in range(latent_dim)]
feat_names.extend(feat_names_fs)

# -----------------------------
# 0) Настройки
# -----------------------------
CSV_PATH = "/content/drive/MyDrive/monkeytype/mt_dataset_pref4feat16.csv"
RANDOM_STATE = 0
TEST_SIZE = 0.2
VAL_SIZE = 0.25  # доля от trainval
BATCH_SIZE = 256
EPOCHS = 150
LR = 3e-4
WEIGHT_DECAY = 1e-3
DROPOUT = 0.5
HIDDEN = (1024, 768)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# 1) Подготовка колонок
# -----------------------------
def cols(prefix: str):
    return [f"{prefix}_{f}" for f in feat_names]

def build_columns(df: pd.DataFrame):
    # location one-hot колонки для каждого x*
    loc_names = ["location_0", "location_1", "location_2", "location_3"]

    loc_cols = {}
    for xi in ["x1", "x2", "x3", "x4"]:
        loc_cols[xi] = [f"{xi}_{n}" for n in loc_names]

    # Y: класс из x1_location_*
    y_cols = loc_cols["x1"]

    missing_y = [c for c in y_cols if c not in df.columns]
    if missing_y:
        raise ValueError(f"Missing target columns (x1 locations): {missing_y}")

    x1_cols = cols("x1")
    x2_cols = cols("x2")
    x3_cols = cols("x3")
    x4_cols = cols("x4")

    # X: все колонки признаков, но исключаем x*_location_0..3
    #all_drop = loc_cols["x1"] + loc_cols["x2"] + loc_cols["x3"] + loc_cols["x4"]

    # Берём только числовые признаки (на всякий)
    #num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    X_cols = x1_cols + x2_cols + x3_cols + x4_cols

    if len(X_cols) == 0:
        raise ValueError("X_cols is empty after dropping location columns. Check your dataframe columns.")

    return X_cols, y_cols


def make_X_y(df: pd.DataFrame, X_cols, y_cols):
    # X
    X = df[X_cols].to_numpy(dtype=np.float32)

    # y: argmax one-hot x1_location_*
    y_onehot = df[y_cols].to_numpy(dtype=np.float32)  # (N,4)

    # sanity: если есть “не one-hot”, всё равно возьмём argmax, но можно проверить суммы
    #sums = y_onehot.sum(axis=1)
    #print("y onehot sums stats:", sums.min(), sums.max(), sums.mean())

    y = y_onehot.argmax(axis=1).astype(np.int64)  # (N,)
    return X, y


# -----------------------------
# 2) Dataset
# -----------------------------
class TabDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# -----------------------------
# 3) MLP-классификатор
# -----------------------------
class MLPClassifier(nn.Module):
    def __init__(self, in_dim: int, hidden=(64, 32), dropout=0.4, n_classes=4):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.LayerNorm(h), nn.Dropout(dropout)]
            d = h
        layers += [nn.Linear(d, n_classes)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def eval_loop(model, loader, device):
    model.eval()
    all_y = []
    all_pred = []
    total_loss = 0.0
    total = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = F.cross_entropy(logits, yb)

        bs = xb.size(0)
        total_loss += float(loss.item()) * bs
        total += bs

        pred = logits.argmax(dim=1)
        all_y.append(yb.detach().cpu().numpy())
        all_pred.append(pred.detach().cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)
    acc = accuracy_score(y_true, y_pred)

    return {
        "loss": total_loss / max(total, 1),
        "acc": acc,
        "y_true": y_true,
        "y_pred": y_pred,
    }


def train_classifier_from_quads(csv_path: str):
    df = pd.read_csv(csv_path)

    X_cols, y_cols = build_columns(df)
    X, y = make_X_y(df, X_cols, y_cols)

    # -----------------------------
    # split: общий test holdout, потом val от trainval
    # -----------------------------
    idx = np.arange(len(df))
    idx_trva, idx_te = train_test_split(
        idx, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True, stratify=y
    )
    idx_tr, idx_va = train_test_split(
        idx_trva, test_size=VAL_SIZE, random_state=RANDOM_STATE, shuffle=True, stratify=y[idx_trva]
    )

    X_tr, y_tr = X[idx_tr], y[idx_tr]
    X_va, y_va = X[idx_va], y[idx_va]
    X_te, y_te = X[idx_te], y[idx_te]

    # scaler только по train
    scaler = StandardScaler()
    scaler.fit(X_tr)
    X_tr = scaler.transform(X_tr).astype(np.float32)
    X_va = scaler.transform(X_va).astype(np.float32)
    X_te = scaler.transform(X_te).astype(np.float32)

    train_loader = DataLoader(TabDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(TabDataset(X_va, y_va), batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(TabDataset(X_te, y_te), batch_size=BATCH_SIZE, shuffle=False)

    model = MLPClassifier(in_dim=X_tr.shape[1], hidden=HIDDEN, dropout=DROPOUT, n_classes=4).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # простой early stopping по val loss (опционально)
    best_state = None
    best_val = float("inf")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            logits = model(xb)
            loss = F.cross_entropy(logits, yb, label_smoothing=0.05)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        trm = eval_loop(model, train_loader, DEVICE)
        vam = eval_loop(model, val_loader, DEVICE)


        print(f"epoch {epoch:03d} | train loss {trm['loss']:.4f} acc {trm['acc']:.3f} | "
                  f"val loss {vam['loss']:.4f} acc {vam['acc']:.3f}")

        if vam["loss"] < best_val: #best_val - 1e-4
            best_val = vam["loss"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    tem = eval_loop(model, test_loader, DEVICE)
    print(f"[TEST] loss {tem['loss']:.4f} acc {tem['acc']:.3f}")

    # дополнительные отчёты
    print("\nClassification report (TEST):")
    print(classification_report(tem["y_true"], tem["y_pred"], digits=3))

    print("Confusion matrix (TEST):")
    print(confusion_matrix(tem["y_true"], tem["y_pred"]))

    info = {
        "X_cols": X_cols,
        "y_cols": y_cols,
        "sizes": {"train": len(idx_tr), "val": len(idx_va), "test": len(idx_te)},
        "indices": {"train": idx_tr, "val": idx_va, "test": idx_te},
        "test_metrics": {"loss": tem["loss"], "acc": tem["acc"]},
    }

    report = eval_test_by_participants_classifier(
        df_test=df.iloc[idx_te].reset_index(drop=True),
        model=model,
        scaler=scaler,
        X_cols=X_cols,
        y_cols=y_cols,
        device=DEVICE,
        batch_size=256,
    )
    print("\n[TEST per participant]")
    print(report)

    report = eval_test_by_participants_classifier(
        df_test=df.iloc[idx_va].reset_index(drop=True),
        model=model,
        scaler=scaler,
        X_cols=X_cols,
        y_cols=y_cols,
        device=DEVICE,
        batch_size=256,
    )
    print("\n[Val per participant]")
    print(report)
    return model, scaler, info

In [17]:
    model, scaler, info = train_classifier_from_quads(CSV_PATH)
    #print("\nInfo:", info)

epoch 001 | train loss 1.0945 acc 0.515 | val loss 1.2226 acc 0.467
epoch 002 | train loss 1.0201 acc 0.557 | val loss 1.2270 acc 0.484
epoch 003 | train loss 0.9427 acc 0.584 | val loss 1.2115 acc 0.460
epoch 004 | train loss 0.8829 acc 0.624 | val loss 1.1921 acc 0.460
epoch 005 | train loss 0.8463 acc 0.640 | val loss 1.1931 acc 0.464
epoch 006 | train loss 0.8143 acc 0.652 | val loss 1.1852 acc 0.477
epoch 007 | train loss 0.7856 acc 0.673 | val loss 1.1895 acc 0.486
epoch 008 | train loss 0.7605 acc 0.689 | val loss 1.1937 acc 0.472
epoch 009 | train loss 0.7346 acc 0.703 | val loss 1.1851 acc 0.473
epoch 010 | train loss 0.7147 acc 0.720 | val loss 1.1852 acc 0.474
epoch 011 | train loss 0.6898 acc 0.735 | val loss 1.1844 acc 0.478
epoch 012 | train loss 0.6679 acc 0.753 | val loss 1.1880 acc 0.470
epoch 013 | train loss 0.6489 acc 0.747 | val loss 1.1884 acc 0.475
epoch 014 | train loss 0.6237 acc 0.772 | val loss 1.1946 acc 0.463
epoch 015 | train loss 0.6023 acc 0.785 | val lo